# <center>5 Ways Your Cross-Validation Lies to You</center>

<center>

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python&logoColor=white)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.3-orange?logo=scikit-learn)
![NumPy](https://img.shields.io/badge/NumPy-1.26-013243?logo=numpy)
![pandas](https://img.shields.io/badge/pandas-2.x-150458?logo=pandas)
![License](https://img.shields.io/badge/License-MIT-red)

</center>

---

**Author:** Lorenzo Scaturchio  
**Last Updated:** July 2026  
**Kernel Version:** 1.0

---

## TL;DR

The most expensive bug in a Kaggle pipeline is not a crash — it is a **local CV
score that looks great and means nothing.** Data leakage lets information from
the validation fold sneak into training, so your CV climbs while your
leaderboard score does not. This notebook builds **five leaks from scratch on
data with no real signal**, shows each one producing an impressive-but-fake
score, then fixes it and watches the score collapse back to honest. Every number
below is computed live — fork it and reproduce the collapse yourself.

| # | The leak | Fake CV | Honest CV |
|---|----------|:-------:|:---------:|
| 1 | Feature selection on the full dataset | ~0.74 | ~0.50 |
| 2 | Target encoding without out-of-fold | ~0.84 | ~0.49 |
| 3 | Random KFold on grouped rows | **1.00** | ~0.50 |
| 4 | Shuffled CV on a time series | R² ~0.99 | R² < 0 |
| 5 | Duplicate rows split across folds | ~0.76 | ~0.55 |

*(Ground truth in every case is chance — ~0.50 AUC / ~0 R². Any lift above that
is the leak talking.)*

## Table of Contents

1. [Objective](#1.-Objective)
2. [What Leakage Is, Precisely](#2.-What-Leakage-Is,-Precisely)
3. [Leak 1 — Preprocessing on the Full Dataset](#3.-Leak-1)
4. [Leak 2 — Target Encoding Without Out-of-Fold](#4.-Leak-2)
5. [Leak 3 — Random Splits on Grouped Data](#5.-Leak-3)
6. [Leak 4 — Shuffled CV on a Time Series](#6.-Leak-4)
7. [Leak 5 — Duplicate Rows Across Folds](#7.-Leak-5)
8. [The Damage, Side by Side](#8.-The-Damage,-Side-by-Side)
9. [A Leak-Proofing Checklist](#9.-A-Leak-Proofing-Checklist)
10. [Conclusion](#10.-Conclusion)

## 1. Objective

Every Kaggler eventually lives the same horror story: local CV says 0.92, you
submit, the leaderboard says 0.78. The gap is almost always **leakage** — and
the insidious part is that a leaky pipeline runs without error and *feels* like
progress.

By the end of this notebook you will be able to:

- name the five leakage patterns that account for most "my CV lied" posts;
- **see each one inflate a score on pure-noise data**, so you trust the
  mechanism rather than taking it on faith;
- fix each with the correct scikit-learn construct (`Pipeline`, out-of-fold
  encoding, `GroupKFold`, `TimeSeriesSplit`, de-duplication);
- apply a pre-submission checklist that catches leakage before the leaderboard
  does.

The trick used throughout: the data has **no real signal**. Targets are random,
or learnable only by cheating. So the honest score *must* be chance — and every
point above chance is the leak, measured.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    cross_val_score, KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

SEED = 0
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# Every leak records (fake, honest) here for the summary chart in Section 8.
SCORES = {}
print("environment ready")

## 2. What Leakage Is, Precisely

> **Leakage:** any path by which information that would not be available at
> prediction time influences training or model selection.

In a cross-validation setting it has one operational meaning: **the validation
fold touched the training process.** That contact can be direct (the same row
appears in both) or indirect (a scaler, a feature, or a selection step was fit
using the validation rows). The result is always the same — the model is
partly graded on data it has already seen, so the CV score is optimistic.

The fix is always the same principle too: **every step that learns from data —
not just the model — must be fit inside the fold, on training rows only.** The
five sections below are five faces of that single rule.

## 3. Leak 1 — Preprocessing on the Full Dataset

The classic. You select the top-K features (or fit a scaler, or a PCA) on the
whole dataset *before* cross-validating the model. The selection step already
peeked at every validation label to decide which features matter.

**Setup:** 800 rows, 5,000 pure-noise features, a random binary target. There
is nothing to learn — honest AUC must be ~0.50.

In [ ]:
n, p = 800, 5000
X = rng.standard_normal((n, p))
y = rng.integers(0, 2, n)

# WRONG: pick the 20 "best" features using the whole y, then CV the model
X_pre = SelectKBest(f_classif, k=20).fit(X, y).transform(X)
fake = cross_val_score(LogisticRegression(max_iter=1000), X_pre, y,
                       cv=5, scoring="roc_auc").mean()

# RIGHT: selection lives inside the pipeline, re-fit on each training fold
pipe = Pipeline([("sel", SelectKBest(f_classif, k=20)),
                 ("clf", LogisticRegression(max_iter=1000))])
honest = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean()

SCORES["Feature selection\non full data"] = (fake, honest)
print(f"leaked AUC  = {fake:.3f}   <- looks like real signal")
print(f"honest AUC  = {honest:.3f}   <- the truth: pure noise")

A ~0.24 AUC gap conjured from **random numbers**. Selecting 20 of 5,000 noise
columns by their correlation with the full target guarantees some will look
predictive on the validation rows they were chosen with. Put the selector in a
`Pipeline` and `cross_val_score` re-fits it per fold — the illusion vanishes.

## 4. Leak 2 — Target Encoding Without Out-of-Fold

Target (mean) encoding replaces a category with the average target for that
category. Done naively — computing each row's encoding from a mean that
**includes that row** — it leaks the label straight into the feature. The
damage scales with cardinality: rare categories essentially memorise their own
targets.

**Setup:** 1,500 rows, a 600-category column (~2-3 rows each), random target.

In [ ]:
n = 1500
cat = rng.integers(0, 600, n)      # high-cardinality id
y = rng.integers(0, 2, n)

# WRONG: encode each row with its category mean over ALL rows (includes itself)
te_full = pd.Series(y).groupby(cat).transform("mean").to_numpy().reshape(-1, 1)
fake = cross_val_score(LogisticRegression(max_iter=1000), te_full, y,
                       cv=5, scoring="roc_auc").mean()

# RIGHT: out-of-fold encoding — each row encoded from OTHER folds only
oof = np.zeros(n)
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(cat, y):
    fold_mean = pd.Series(y[tr]).groupby(cat[tr]).mean()
    oof[va] = pd.Series(cat[va]).map(fold_mean).fillna(y[tr].mean()).to_numpy()
honest = cross_val_score(LogisticRegression(max_iter=1000), oof.reshape(-1, 1), y,
                         cv=5, scoring="roc_auc").mean()

SCORES["Target encoding\nwithout OOF"] = (fake, honest)
print(f"leaked AUC  = {fake:.3f}   <- the feature is a disguised copy of y")
print(f"honest AUC  = {honest:.3f}   <- out-of-fold encoding tells the truth")

The naive encoding scored ~0.84 on a **random** target — because for a category
with two rows, "mean target of this category" is almost literally this row's
label. Out-of-fold encoding computes each row's value from folds that exclude
it, and the fake signal disappears. (Smoothing toward the global mean helps too,
but out-of-fold is the non-negotiable part.)

## 5. Leak 3 — Random Splits on Grouped Data

When rows cluster into groups — multiple visits per patient, several photos per
user, repeated measurements per device — and the label is a property of the
**group**, a random KFold puts some of a group's rows in train and the rest in
validation. The model memorises the group identity and reads the answer off.

**Setup:** 150 users x 10 rows; the label is constant within a user and random
across users; the *only* feature is the one-hot user id. Nothing generalises —
honest AUC is 0.50.

In [ ]:
groups = np.repeat(np.arange(150), 10)
user_label = rng.integers(0, 2, 150)
y = user_label[groups]
X_id = OneHotEncoder(sparse_output=False).fit_transform(groups.reshape(-1, 1))

# WRONG: random KFold — a user's rows land in both train and validation
fake = cross_val_score(LogisticRegression(max_iter=2000), X_id, y,
                       cv=KFold(5, shuffle=True, random_state=SEED),
                       scoring="roc_auc").mean()

# RIGHT: GroupKFold keeps every user entirely on one side of the split
honest = cross_val_score(LogisticRegression(max_iter=2000), X_id, y,
                         groups=groups, cv=GroupKFold(5),
                         scoring="roc_auc").mean()

SCORES["Random KFold\non grouped rows"] = (fake, honest)
print(f"leaked AUC  = {fake:.3f}   <- perfect score by memorising user id")
print(f"honest AUC  = {honest:.3f}   <- unseen users are a coin flip")

A flawless **1.000** versus a coin-flip **0.500** — the widest gap in the
notebook, and the one that most often survives into real pipelines because the
code looks completely normal. If your rows have any entity behind them —
user, session, image series, molecule — ask whether that entity should be a
`groups=` argument. When in doubt, it should.

## 6. Leak 4 — Shuffled CV on a Time Series

On temporal data, a shuffled KFold lets the model train on **future** points to
predict the **past** — it interpolates between validation neighbours instead of
forecasting. The score looks superb and is meaningless for anything you would
actually deploy (which only ever sees the past).

**Setup:** a smooth seasonal + trend series with the time index as the only
feature. Shuffled CV can interpolate the curve; honest forward-chaining CV must
extrapolate it.

In [ ]:
m = 1200
t = np.arange(m)
y = np.sin(t / 30.0) * 5 + 0.01 * t + rng.standard_normal(m) * 0.4
X_t = t.reshape(-1, 1).astype(float)

def gb():
    return GradientBoostingRegressor(n_estimators=120, max_depth=3, random_state=SEED)

# WRONG: shuffled KFold — validation points sit between training points in time
fake = cross_val_score(gb(), X_t, y,
                       cv=KFold(5, shuffle=True, random_state=SEED),
                       scoring="r2").mean()

# RIGHT: TimeSeriesSplit — always train on the past, validate on the future
honest = cross_val_score(gb(), X_t, y, cv=TimeSeriesSplit(5), scoring="r2").mean()

SCORES["Shuffled CV\non time series"] = (fake, honest)
print(f"shuffled R2 = {fake:.3f}   <- interpolating between future & past")
print(f"forward  R2 = {honest:.3f}   <- honest forecasting is far harder")

Shuffled CV reports a near-perfect R² ~0.99; the honest forward split reports a
**negative** R² — the model genuinely struggles to extrapolate the trend, which
is the real difficulty of forecasting. The shuffled number is not "slightly
optimistic," it is describing a task you will never actually face. Any data with
a time axis needs `TimeSeriesSplit` (or a manual date cutoff).

## 7. Leak 5 — Duplicate Rows Across Folds

Duplicates and near-duplicates are everywhere: repeated records, augmented
copies, the same event logged twice. When a row and its copy land on opposite
sides of a random split, the model has literally seen the validation answer.
High-capacity models (KNN, trees) exploit it hardest.

**Setup:** 500 random rows, each triplicated, shuffled. A 5-NN classifier finds
a row's own copies among its nearest neighbours.

In [ ]:
n0 = 500
X_base = rng.standard_normal((n0, 20))
y_base = rng.integers(0, 2, n0)
X_dup = np.vstack([X_base, X_base, X_base])          # each row appears 3x
y_dup = np.concatenate([y_base, y_base, y_base])
order = rng.permutation(len(y_dup))
X_dup, y_dup = X_dup[order], y_dup[order]

# WRONG: random split — copies of a row straddle the fold boundary
fake = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_dup, y_dup,
                       cv=KFold(5, shuffle=True, random_state=SEED),
                       scoring="roc_auc").mean()

# RIGHT: de-duplicate before validating
_, uniq = np.unique(X_dup, axis=0, return_index=True)
honest = cross_val_score(KNeighborsClassifier(n_neighbors=5),
                         X_dup[uniq], y_dup[uniq], cv=5, scoring="roc_auc").mean()

SCORES["Duplicate rows\nacross folds"] = (fake, honest)
print(f"leaked AUC  = {fake:.3f}   <- neighbours are its own copies")
print(f"honest AUC  = {honest:.3f}   <- deduped, back to chance")

The duplicated set scores ~0.76 on noise because most validation rows have an
identical twin sitting in the training fold. De-duplicating drops it back to
~0.55 — essentially chance, since a 500-row noise AUC naturally wobbles a few
points around 0.50 (the leak, ~0.20 of AUC, is what actually mattered). Always
check `df.duplicated().sum()` — and be suspicious of near-duplicates from
augmentation or logging too.

## 8. The Damage, Side by Side

One chart from the five results above. Each pair shows the fake score the leak
produced against the honest score after the fix. The gap between them is,
literally, fiction — and it is exactly what evaporates on the leaderboard.

In [ ]:
labels = list(SCORES.keys())
fake_scores = [SCORES[k][0] for k in labels]
honest_scores = [SCORES[k][1] for k in labels]

y_pos = np.arange(len(labels))
h = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.barh(y_pos + h/2, fake_scores, height=h, color="#D64550",
             label="Leaked (fake)", zorder=3)
b2 = ax.barh(y_pos - h/2, honest_scores, height=h, color="#2E7CD6",
             label="Honest (fixed)", zorder=3)
ax.bar_label(b1, fmt="%.2f", padding=4, fontsize=9)
ax.bar_label(b2, fmt="%.2f", padding=4, fontsize=9)
ax.axvline(0.5, color="#555555", linewidth=1, linestyle="--", zorder=2)
ax.text(0.5, len(labels) - 0.4, " chance (AUC 0.5)", color="#666666", fontsize=9)
ax.set_yticks(y_pos, labels)
ax.invert_yaxis()
ax.set_xlabel("Cross-validation score (AUC; Leak 4 uses R\u00b2 and is off this scale)")
ax.set_title("Five leaks: the fake score vs the honest score", loc="left")
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#DDDDDD", linewidth=0.6, zorder=0)
plt.tight_layout()
plt.show()

print("Leak                              fake     honest    gap")
for k in labels:
    f, hs = SCORES[k]
    tag = k.replace(chr(10), " ")
    print(f"{tag:<34s}{f:6.3f}   {hs:6.3f}   {f - hs:+.3f}")

Note the chart mixes AUC (Leaks 1-3, 5) with the time-series R² (Leak 4), so
read the time-series bar qualitatively — the point is direction and magnitude,
not a shared scale. Every red bar stands well above the 0.5 chance line while
its blue partner sits on it: **the entire height difference was leakage.**

## 9. A Leak-Proofing Checklist

Run this before you trust a CV number:

- [ ] **Is every fitted step inside the fold?** Scalers, imputers, selectors,
      PCA, encoders — wrap them in a `Pipeline` so `cross_val_score` re-fits
      them per fold. Nothing that calls `.fit` should touch the full dataset.
- [ ] **Any target-derived feature computed out-of-fold?** Target/count/
      likelihood encodings must be built with an out-of-fold scheme, never a
      whole-column `groupby`.
- [ ] **Do rows have a hidden entity?** User, session, patient, image-series,
      molecule — if the label is a property of that entity, use `GroupKFold`
      (or `StratifiedGroupKFold`).
- [ ] **Is there a time axis?** Use `TimeSeriesSplit` or a hard date cutoff;
      never shuffle. Also confirm no feature (rolling mean, "days until X")
      secretly uses the future.
- [ ] **Duplicates checked?** `df.duplicated().sum()`, plus a look for
      near-duplicates from augmentation or double-logging.
- [ ] **Sanity test:** does a CV score above chance survive on a **shuffled
      target**? If a model "predicts" random labels, you have a leak. This one
      check would have caught all five leaks above.
- [ ] **Does local CV track the leaderboard?** If they move together across a
      few submissions, trust CV. If CV climbs while the board does not, stop and
      hunt for leakage before tuning anything else.

## 10. Conclusion

**Takeaways**

1. Leakage is one rule broken five ways: **something that learns from data saw
   the validation rows.** Fix it by fitting every such step inside the fold.
2. On data with zero real signal, each leak manufactured a convincing score —
   up to a perfect 1.00 — and each fix collapsed it back to chance. The gap was
   never skill; it was information bleed.
3. The **shuffled-target test** is the cheapest insurance you can buy: if a
   pipeline scores above chance on random labels, it leaks.
4. Trust CV only once it *moves with* the leaderboard. An honest 0.50 beats a
   fake 0.90, because you can actually improve on the honest one.

**Next experiments to try on your own**

- Add the shuffled-target check as an assertion in your own CV harness.
- Re-run Leak 3 with `StratifiedGroupKFold` to keep class balance *and* group
  integrity at once.
- Take a past competition where your CV and LB diverged and diagnose which of
  these five (or which leaky feature) was responsible.

**Related notebooks in this series:**

- Feature Engineering Cookbook: 50 Techniques
- Optuna Tuning: A Practical Kaggle Guide
- Polars on Kaggle: The Complete Speed Guide

---

**If this notebook saved you a leaderboard faceplant, please upvote!** Questions
and war stories welcome in the comments.

*Lorenzo Scaturchio | July 2026*